# Neural PointMaze — Full Training (Colab)

Runs the full 5000-episode 2-phase training with stabilization improvements:
- Phase-aware epsilon reset at Phase 1→2 boundary
- Buffer truncation (remove stale diverse data)
- Cosine LR annealing with warm restart
- Gradient clipping on reward network

**Requires gymnasium-robotics** (installed automatically below).

**Runtime**: ~1-2 hours on CPU, ~30-45 min with GPU

In [ ]:
# 1. Install dependencies and clone repo
!pip install -q gymnasium-robotics mujoco torch numpy scipy matplotlib scikit-learn imageio imageio-ffmpeg

# Clone repo
!git clone -b neural-successor-representation https://github.com/PrashRangarajan/Successor_Active_Inference_Clean.git
%cd Successor_Active_Inference_Clean

In [ ]:
# 2. Verify imports and PointMaze environment
import sys
sys.path.insert(0, '.')

import torch
import gymnasium as gym
import gymnasium_robotics
import numpy as np

gym.register_envs(gymnasium_robotics)

# Test PointMaze environment
env = gym.make('PointMaze_UMaze-v3')
obs, _ = env.reset()
print(f'Observation keys: {list(obs.keys())}')
print(f'observation shape: {obs["observation"].shape}')
print(f'desired_goal shape: {obs["desired_goal"].shape}')
print(f'Action space: {env.action_space}')
env.close()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# 3. Run full training (5000 episodes: 2000 diverse + 3000 mixed)
import torch
device_flag = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device_flag}')
!python -u examples/run_neural_point_maze.py --device {device_flag}

In [ ]:
# 4. Inspect training diagnostics
import torch
import numpy as np

checkpoint = torch.load('data/neural_point_maze/checkpoint.pt',
                        map_location='cpu', weights_only=False)
print('Checkpoint keys:', list(checkpoint.keys()))
print(f'Total steps: {checkpoint["total_steps"]}')
print(f'Final epsilon: {checkpoint["epsilon"]:.4f}')

In [ ]:
# 5. Plot training curves
import matplotlib.pyplot as plt
import numpy as np

log = checkpoint.get('training_log', {})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (key, title) in zip(axes.flat, [
    ('sf_loss', 'SF TD Loss'),
    ('reward_loss', 'Reward Prediction Loss'),
    ('episode_reward', 'Episode Reward'),
    ('episode_steps', 'Episode Steps'),
]):
    data = log.get(key, [])
    if data:
        ax.plot(data, alpha=0.3, linewidth=0.5)
        w = min(100, len(data) // 10 + 1)
        if w > 1:
            sm = np.convolve(data, np.ones(w)/w, mode='valid')
            ax.plot(np.arange(w-1, w-1+len(sm)), sm, linewidth=2)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Download checkpoints and results
from google.colab import files
import shutil

shutil.make_archive('neural_pointmaze_results', 'zip', '.', 'data/neural_point_maze')
files.download('neural_pointmaze_results.zip')